In [17]:
from functions import ping_openai, read_prompt_from_file_only, load_function_schema, normalize_id, normalize_type, combine_paragraphs, format_nodes_for_prompt, get_schema, setup_logger
from functions import call_openai_function
from model_dictionary import model_dictionary
from inputs import relationship_groups, groups_to_prompts, nodes_by_group_prompt, characteristic_node_types, required_node_types
from openai_client import openai_client

import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from openai import OpenAI
from dotenv import load_dotenv
import json
from bson import json_util
import re
from datetime import datetime, timezone
load_dotenv()

# MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("MONGO_COLLECTION_NAME")

if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# establish MongoDB client
try:
    mongo_client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = mongo_client[DB_NAME]

    # Define collections
    articles_collection = db[ARTICLES_COLLECTION_NAME]  # Stores articles

    # Verify connection
    mongo_client.admin.command('ping')
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

# establish openai client 
ping_openai(openai_client) 

✅ Connected to MongoDB Atlas! Database: tuone
✅ Successfully connected to OpenAI API!
Available Models: ['gpt-4o-realtime-preview-2024-12-17', 'gpt-4o-audio-preview-2024-12-17', 'gpt-4-1106-preview', 'dall-e-3', 'dall-e-2', 'gpt-4o-audio-preview-2024-10-01', 'gpt-4-turbo-preview', 'text-embedding-3-small', 'babbage-002', 'gpt-4', 'text-embedding-ada-002', 'chatgpt-4o-latest', 'gpt-4o-mini-audio-preview', 'gpt-4o-audio-preview', 'o1-preview-2024-09-12', 'gpt-4o-mini-realtime-preview', 'gpt-4o-mini-realtime-preview-2024-12-17', 'gpt-4.1-nano', 'gpt-3.5-turbo-instruct-0914', 'gpt-4o-mini-search-preview', 'gpt-4.1-nano-2025-04-14', 'gpt-3.5-turbo-16k', 'gpt-4o-realtime-preview', 'davinci-002', 'gpt-3.5-turbo-1106', 'gpt-4o-search-preview', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo', 'o3-mini-2025-01-31', 'gpt-4o-mini-search-preview-2025-03-11', 'gpt-4-0125-preview', 'gpt-4o-2024-11-20', 'gpt-4o-2024-05-13', 'text-embedding-3-large', 'o1-2024-12-17', 'o1', 'o1-preview', 'gpt-4-0613', 'o1-min

In [18]:
def extract_nodes(article_id, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_name, logger):

    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    user_content = f"Here is the article: {text}"

    nodes_by_category = call_openai_function(
        prompt=prompt,
        user_content=user_content,
        function_schema=function_schema,
        function_name="extract_clean_tech_entities",
        expected_top_key="nodes",
        model_name=model_name,
        logger=logger
    )

    # Flatten nodes while retaining their type
    formatted_nodes = []
    for node_type, node_categories in nodes_by_category.items():
        for node in node_categories:
            formatted_node = {
                "article_id": article_id,
                "id": normalize_id(node.get("id")),
                "type": normalize_type(node.get("type")),
                "name": node.get("name"),
                "location": {
                    "city": node.get("location", {}).get("city", ""),
                    "country": node.get("location", {}).get("country", "")
                } if node.get("location") else None,
                "amount": node.get("amount"), #possibly to drop
                "status": node.get("status"),
            }
            formatted_nodes.append(formatted_node)

    return formatted_nodes

In [19]:
def extract_node_characteristics(article_id, text, nodes, relationship_group, model_name, logger):

    PROMPT_PATH = groups_to_prompts[relationship_group]
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    FUNCTION_SCHEMA_PATH = f"schemas/{relationship_group}.json"
    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)

    allowed_types = nodes_by_group_prompt[relationship_group]
    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)
    
    user_content = f"""Here is the article text: {text}
    Here is the list of known entities:
    {compact_nodes}
    """

    return call_openai_function(
        prompt=prompt,
        user_content=user_content,
        function_schema=function_schema,
        function_name="extract_characteristics",
        expected_top_key="node_characteristics",
        model_name=model_name,
        logger=logger
    )

In [23]:
def extract_relationships(article_id, text, nodes, relationship_group, model_name, logger, allowed_types=None):
    PROMPT_PATH = groups_to_prompts[relationship_group]
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    FUNCTION_SCHEMA_PATH = f"schemas/{relationship_group}.json"
    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)

    allowed_types = nodes_by_group_prompt[relationship_group]
    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)

    user_content = f"""Here is the article text: {text}
    Here is the list of known entities:
    {compact_nodes}
    Please extract only the specified relationship types.
    """

    # Use shared OpenAI wrapper
    raw_relationships = call_openai_function(
        prompt=prompt,
        user_content=user_content,
        function_schema=function_schema,
        function_name="extract_clean_tech_relationships",
        expected_top_key="relationships",
        model_name=model_name,
        logger=logger
    )

    # Format output
    formatted_relationships = []
    for rel in raw_relationships:
        raw_source = rel.get("source")
        raw_target = rel.get("target")
        norm_source = normalize_id(raw_source)
        norm_target = normalize_id(raw_target)

        formatted_relationships.append({
            "article_id": article_id,
            "id": f"{norm_source}_{rel.get('type')}_{norm_target}",
            "source": norm_source,
            "target": norm_target,
            "type": rel.get("type"),
            "group": relationship_group
        })

    return formatted_relationships

In [24]:
def process_articles(articles_to_process, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_dictionary):

    for article in articles_to_process:
        articleID  = str(article["_id"])
        friendly_id = article["meta"]["ID"]
        logger      = setup_logger(friendly_id)

        print(f"📌 Processing Article: {article['title']}")
        logger.info("📌 Processing Article ID: %s — %s", friendly_id, article["title"])

        # ── 1. Skip if already validated ──────────────────────────────────────────
        val = article.get("validation")
        if val is True:
            print("⏭️  Skipping – article is validated")
            logger.info("⏭️  Skipping – article is validated")
            continue

        if isinstance(val, (int, float)):        # timestamp → already processed
            processed_on = datetime.fromtimestamp(val, tz=timezone.utc)\
                                     .strftime("%Y-%m-%d %H:%M UTC")
            print(f"⏭️  Skipping – article was validated on {processed_on}")
            logger.info("⏭️  Skipping – article was validated on %s", processed_on)
            continue

        # ── 2. Gather text; skip if empty ─────────────────────────────────────────
        text = combine_paragraphs(article)
        if not text:
            logger.warning("⚠️  No valid text found; skipping.")
            continue

        try:
            # extract entities
            formatted_nodes = extract_nodes(articleID, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_dictionary["nodes"], logger)
            logger.info(f"Formatted nodes used for all prompts hereafter: {formatted_nodes}")

            # attach characteristics to entities (capacities and investments)
            for relationship_group, config in characteristic_node_types.items():
                model_name = model_dictionary[relationship_group] # select fine-tuned model
                id_key = config["id_key"]
                type_match = config["type_match"]

                # only continue if there are nodes of this type in the article
                has_relevant_nodes = any(node.get("type") == type_match for node in formatted_nodes)
                if not has_relevant_nodes:
                    logger.info(f"⏭️ Skipping {relationship_group} – no nodes of type '{type_match}' found.")
                    continue

                logger.info(f"🔍 Extracting characteristics for node group: {relationship_group}")
                node_characteristics = extract_node_characteristics(articleID, text, formatted_nodes, relationship_group, model_name, logger)  

                # attach 'status' and 'type' to matching capacity nodes (flat list structure)
                for char in node_characteristics:
                    node_id = char.get(id_key)
                    logger.info(f"Node ID is {node_id}")
                    found_match = False

                    for node in formatted_nodes:
                        if node.get("id") == node_id and node.get("type") == type_match:
                            logger.info(f"✅ Match found for {id_key} '{node_id}' in {type_match} nodes")
                            if "status" in char:
                                node["status"] = char["status"]
                                logger.info(f"  ➕ Set status: {char['status']}")
                            if "phase" in char:
                                node["phase"] = char["phase"]
                                logger.info(f"  ➕ Set phase: {char['phase']}")
                            found_match = True

                    if not found_match:
                        logger.info(f"⚠️ No match found for {id_key} '{node_id}' in formatted_nodes")

            # extract relationships between entities
            all_relationships = []
            for relationship_group,model_name in model_dictionary.items():
                if relationship_group in ["nodes", "capacities", "investments"]: #skip nodes, capacities and investments which have logic elsewhere
                    continue

                def has_required_nodes(formatted_nodes, required_types):
                    existing_types = {node['type'] for node in formatted_nodes}
                    return any(req_type in existing_types for req_type in required_types)
                
                required_types = required_node_types.get(relationship_group, [])
                if not has_required_nodes(formatted_nodes, required_types):
                    logger.info(f"🛑 Skipping {relationship_group}: required nodes {required_types} not present.")
                    continue

                logger.info(f"- - - Querying openai for relationships: {relationship_group}, using model: {model_name}")

                allowed_types = relationship_groups[relationship_group]
                logger.info(f"Node types included in the query: {allowed_types}")

                relationships = extract_relationships(articleID, text, formatted_nodes, relationship_group, model_name, logger, allowed_types)
                all_relationships.extend(relationships)

            # update entries in mongodb with clean entities and relationships
            update_result = articles_collection.update_one(
                {"_id": article["_id"]},  #match by article id
                {"$set": {
                    "nodes": formatted_nodes or [],
                    "relationships": all_relationships or []
                    }})

            if update_result.modified_count > 0:
                logger.info(f"✅ Updated Article ID: {articleID} with {len(formatted_nodes)} nodes and combined text.")
            else:
                logger.info(f"⚠️ No updates made for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing Article ID {articleID}: {e}")  

In [25]:
n_articles = 3
offset_articles = 0

articles_to_process = list(
    articles_collection.find(
        {"meta.ID": {"$regex": "^11"}},  
        {"_id": 1, "meta": 1,
        "paragraphs": 1, "validation": 1, "title": 1}       
    )
    .sort("_id", -1)            # sort by MongoDB ObjectId (descending)
    .skip(offset_articles)      # skip first `offset` articles
    .limit(n_articles)          # limit the number of articles
)

PROMPT_PATH = "prompts/entities-only.txt"
FUNCTION_SCHEMA_PATH = "schemas/entities.json"

process_articles(articles_to_process, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_dictionary)

📌 Processing Article: Volkswagen to cut battery production plans in Salzgitter
⏭️  Skipping – article is validated
📌 Processing Article: Mazda to build battery module plant in Iwakuni
⏭️  Skipping – article is validated
📌 Processing Article: Sony-Honda alliance prices electric saloon Afeela 1 at $90,000
